# Proyect Description:
Developed an end-to-end retail analytics and machine learning project focused on customer behavior and high-value customer prediction using transactional supermarket data.

Performed data cleaning, feature engineering, RFM and ABC analysis, and built interactive Power BI dashboards to identify customer patterns, product performance, and branch-level sales insights.

Designed a predictive modeling pipeline using temporal validation strategies to prevent data leakage and transform historical customer behavior into actionable predictive insights.

The project evolved from Customer Lifetime Value (CLV) regression into a High-Value Customer Classification model after evaluating data consistency and business applicability. Implemented imbalance handling, threshold tuning, and precision-recall optimization to improve minority-class detection and business-oriented decision making.


# 1- Data upload

In [1]:
import numpy as np
import pandas as pd

df= pd.read_csv('/content/final_df.csv')
df.head()

,ID_Cliente,Recency,Frequency,Monetary,Avg_Ticket,CLV_60D,Log_CLV_60D,Avg_Purchase_Interval
0,1000,35,2,920404,460202.0,120196.0,11.696887,50.00
1,1001,41,5,1244040,248808.0,0.0,0.000000,14.25
2,1002,30,2,831268,415634.0,0.0,0.000000,46.00
3,1003,41,2,407090,203545.0,64158.0,11.069120,48.00
4,1004,111,1,721545,721545.0,0.0,0.000000,999.00


# 2- Check columns

In [2]:
df.columns

Index(['ID_Cliente', 'Recency', 'Frequency', 'Monetary', 'Avg_Ticket',
       'CLV_60D', 'Log_CLV_60D', 'Avg_Purchase_Interval'],
      dtype='object')

In [3]:
#Drop the feature Log_CLV_60D
#Is not useful for the model
df=df.drop(['Log_CLV_60D'], axis=1)

# 3-Create High Value threshold

In [4]:
threshold = df['CLV_60D'].quantile(0.80)

threshold

np.float64(650872.2000000001)

# 4- Create binary target

In [5]:
df['High_Value_Customer'] = (
    df['CLV_60D'] >= threshold
).astype(int)

In [6]:
df.head()

,ID_Cliente,Recency,Frequency,Monetary,Avg_Ticket,CLV_60D,Avg_Purchase_Interval,High_Value_Customer
0,1000,35,2,920404,460202.0,120196.0,50.00,0
1,1001,41,5,1244040,248808.0,0.0,14.25,0
2,1002,30,2,831268,415634.0,0.0,46.00,0
3,1003,41,2,407090,203545.0,64158.0,48.00,0
4,1004,111,1,721545,721545.0,0.0,999.00,0


#Step 5: Review the feature balance

In [7]:
df['High_Value_Customer'].value_counts()

,count
High_Value_Customer,
0,6402
1,1601


In [8]:
#Propotions
df['High_Value_Customer'].value_counts(normalize=True)

,proportion
High_Value_Customer,
0,0.79995
1,0.20005


# Step 6: Create X and y

In [9]:
X = df.drop(
    columns=['ID_Cliente','CLV_60D', 'High_Value_Customer']
)
y = df['High_Value_Customer']


# Step 7: Check Features

In [10]:
X.columns

Index(['Recency', 'Frequency', 'Monetary', 'Avg_Ticket',
       'Avg_Purchase_Interval'],
      dtype='object')

In [11]:
X.head()

,Recency,Frequency,Monetary,Avg_Ticket,Avg_Purchase_Interval
0,35,2,920404,460202.0,50.00
1,41,5,1244040,248808.0,14.25
2,30,2,831268,415634.0,46.00
3,41,2,407090,203545.0,48.00
4,111,1,721545,721545.0,999.00


#Step 8: Train - Test Split


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Model Training

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_classifier.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=200, n_jobs=-1,
                       random_state=42)

#Step 10: Predictions

In [14]:
y_pred = rf_classifier.predict(X_test)

#Step 11: Evaluate

In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')

Accuracy: 0.7988757026858213
Precision: 0.25
Recall: 0.003125
F1 Score: 0.006172839506172839


# Metrics report

In [21]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      1.00      0.89      1281
           1       0.25      0.00      0.01       320

    accuracy                           0.80      1601
   macro avg       0.53      0.50      0.45      1601
weighted avg       0.69      0.80      0.71      1601



## From the metrics we can see that is taking the “safe” route:

**“If I predict mostly zeros, I get high accuracy.”**

Simply because most clients are "Class 0"

##The other problem:
**Recall = 0**
###Means: The model almost never detects High Value Customers and this is business-critical class

#*All this is due to the class imbalance*

# But...I can fix it :

In [22]:
rf_classifier = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

#Why this helps

##It tells the model:

**“Class 1 is more important.
Don’t ignore it.”**

# I must repeat the next steps and chech:

In [24]:
#Fit the model
rf_classifier.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_estimators=200,
                       n_jobs=-1, random_state=42)

In [25]:
#Predict
y_pred = rf_classifier.predict(X_test)

In [26]:
#Evaluate with metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')

Accuracy: 0.7026858213616489
Precision: 0.19047619047619047
Recall: 0.15
F1 Score: 0.16783216783216784


In [27]:
  print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.84      0.82      1281
           1       0.19      0.15      0.17       320

    accuracy                           0.70      1601
   macro avg       0.49      0.50      0.49      1601
weighted avg       0.68      0.70      0.69      1601



#Conclusion

BEFORE balancing
Metric	Result:
* Recall	~0.00
* F1	    ~0.01

The model basically ignored: High Value Customers.

NOW:      Metric	Result
* Recall	    -0.15
* F1	        -0.17
* Precision	  -0.19

#That means: The model is finally detecting some valuable customers.

# Now the next logical step to improve de model's performance is changing to another model like XGboost or threshold tuning...

## I will go with the second option this time

# Step 1: Get predictions probabilities

In [28]:
y_probs = rf_classifier.predict_proba(X_test)[:,1]

#This for every customer, returns something like this:

#Customer Id	    Probability High Value
#   4453	                0.72
#   2040	                0.18
#   8920	                0.44

# STEP 2 — Apply custom threshold.
##En this case (0.30)

In [32]:
threshold = 0.30

y_pred_custom = (y_probs >= threshold).astype(int)


# STEP 3 — Evaluate again

In [33]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred_custom)

precision = precision_score(y_test, y_pred_custom)

recall = recall_score(y_test, y_pred_custom)

f1 = f1_score(y_test, y_pred_custom)

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')

Accuracy: 0.21424109931292942
Precision: 0.19545454545454546
Recall: 0.940625
F1 Score: 0.32365591397849464


# And again:

In [34]:
print(classification_report(
    y_test,
    y_pred_custom
))

              precision    recall  f1-score   support

           0       0.69      0.03      0.06      1281
           1       0.20      0.94      0.32       320

    accuracy                           0.21      1601
   macro avg       0.44      0.49      0.19      1601
weighted avg       0.59      0.21      0.11      1601



# What happened technically:

## By lowering the threshold to: 0.30

#The model became: Much more aggressive predicting class 1.

## Meaning:

###*“I’d rather risk false positives than miss valuable customers.”*

## And the metrics confirm it:
* Metric
* Recall: Went from 0.15	   to ---> 0.94
* F1: Went from 0.17 to---> 0.32
* Accuracy: Went from	0.70 to- --> 0.21
* Precision: Went from 0.19 to ---> 0.20

# And this is very common core business tradeoff
	     
* High Recall means, catching almost all valuable customers
* High Precision means, avoiding wasting marketing resources

#For example:
### A company may say: “We prefer contacting 100 customers if 20 are truly valuable.”

#Why?
### Because: Missing a future premium customer could cost much more.

# I will try with other threshold

In [38]:
#thershhold limit
 threshold = 0.40

y_pred_custom2 = (y_probs >= threshold).astype(int)

In [39]:
#Metrics on new boundary
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy = accuracy_score(y_test, y_pred_custom2)

precision = precision_score(y_test, y_pred_custom2)

recall = recall_score(y_test, y_pred_custom2)

f1 = f1_score(y_test, y_pred_custom2)

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')

Accuracy: 0.33041848844472205
Precision: 0.1987179487179487
Recall: 0.775
F1 Score: 0.3163265306122449


In [41]:
#Check
print(classification_report(
    y_test,
    y_pred_custom2
))

              precision    recall  f1-score   support

           0       0.80      0.22      0.34      1281
           1       0.20      0.78      0.32       320

    accuracy                           0.33      1601
   macro avg       0.50      0.50      0.33      1601
weighted avg       0.68      0.33      0.34      1601



# This seems much more balanced
* Why do I say that?:

Look at the tradeoff now:

* Metric:	      Threshold 0.30	/ Threshold 0.40
* Recall: Went from 0.94	to ---> 0.78
* F1: Remained at 0.32 ---> 0.32
* Accuracy: Went from 0.21	to ---> 0.33

# And the key metrics is the Recall:
* Recall = 0.775

###*The model still catching: 78% of High Value Customers.*

##That’s actually pretty solid considering:

* Limited historical depth
* Relatively small feature set
* Noisy retail behavior